# WildTrace Pipeline Driver

This notebook is the current driver for the agentic WildTrace pipeline. It runs the DAG-style stage scripts in order so we can inspect outputs while keeping the same script boundaries that Airflow will use later.

Before running the notebook:
- run `uv sync` from the repo root
- start Ollama with `ollama serve`
- pull the configured local models
- make sure BioCLIP runs on a CUDA-capable GPU machine for bronze enrichment
- create `.env` from `.env.example` if you need custom model or host settings


In [ ]:
from pathlib import Path
import json
import subprocess

import yaml
from IPython.display import SVG, display
from PIL import Image

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "scripts").exists():
    for parent in REPO_ROOT.parents:
        if (parent / "scripts").exists() and (parent / "configs").exists():
            REPO_ROOT = parent
            break

SCRIPTS_DIR = REPO_ROOT / 'scripts'
CONFIG_DIR = REPO_ROOT / 'configs'
PYTHON = REPO_ROOT / '.venv' / 'bin' / 'python'

if not PYTHON.exists():
    raise FileNotFoundError(f"{PYTHON} not found. Run `uv sync` from {REPO_ROOT} first.")

print("repo_root =", REPO_ROOT)
print("python =", PYTHON)
print("scripts_dir =", SCRIPTS_DIR)
print("config_dir =", CONFIG_DIR)


In [ ]:
def run_stage(script_name: str):
    script = SCRIPTS_DIR / script_name
    cmd = [str(PYTHON), str(script), '--repo-root', str(REPO_ROOT), '--config-dir', str(CONFIG_DIR)]
    print('Running:', ' '.join(cmd))
    return subprocess.run(cmd, check=True)


def load_ndjson(path: Path):
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

## Inspect Configs

Tune these first before running a larger fetch.

In [ ]:
for name in ['datasets.yaml', 'storage.yaml', 'quality.yaml', 'models.yaml', 'export.yaml']:
    path = CONFIG_DIR / name
    print(f'\n## {name}')
    print(yaml.safe_dump(yaml.safe_load(path.read_text()), sort_keys=False))

## Step 1: Fetch Open Images

In [ ]:
run_stage('fetch_openimages.py')

fetch_ledger = REPO_ROOT / 'raw_data' / 'bronze' / 'manifests' / 'openimages_fetch.ndjson'
fetch_latest = REPO_ROOT / 'raw_data' / 'bronze' / 'manifests' / 'openimages_fetch_latest.ndjson'
fetch_rows = load_ndjson(fetch_ledger)
fetch_latest_rows = load_ndjson(fetch_latest)
print('fetch ledger rows =', len(fetch_rows))
print('fetch latest rows =', len(fetch_latest_rows))
fetch_latest_rows[:3]

## Step 2: Curate Bronze

In [ ]:
run_stage('curate_bronze.py')

bronze_curated = REPO_ROOT / 'raw_data' / 'bronze' / 'manifests' / 'bronze_curated.ndjson'
bronze_accepted = REPO_ROOT / 'raw_data' / 'bronze' / 'manifests' / 'bronze_accepted.ndjson'
curated_rows = load_ndjson(bronze_curated)
accepted_rows = load_ndjson(bronze_accepted)
print('bronze curated rows =', len(curated_rows))
print('bronze accepted rows =', len(accepted_rows))
accepted_rows[:3]

## Step 3: Normalize To Silver

In [ ]:
run_stage('normalize_to_silver.py')

silver_path = REPO_ROOT / 'processed' / 'silver' / 'qa' / 'silver_samples.ndjson'
silver_rows = load_ndjson(silver_path)
silver_rows[:2]

## Step 4: Enrich And Crop Subjects

In [ ]:
run_stage('enrich_and_crop_subjects.py')

subjects_path = REPO_ROOT / 'processed' / 'silver' / 'qa' / 'silver_subjects.ndjson'
subject_rows = load_ndjson(subjects_path)
subject_rows[:2]

In [ ]:
if subject_rows:
    sample = subject_rows[0]
    display(Image.open(REPO_ROOT / sample['crop_path']))
    if sample['isolated_path']:
        display(Image.open(REPO_ROOT / sample['isolated_path']))

## Step 5: Generate Line Diagrams

In [ ]:
run_stage('generate_line_diagrams.py')

diagram_attempts_path = REPO_ROOT / 'processed' / 'silver' / 'qa' / 'line_diagram_attempts.ndjson'
diagram_attempt_rows = load_ndjson(diagram_attempts_path)
diagram_attempt_rows[:2]

## Step 6: Validate And Retry Diagrams

In [ ]:
run_stage('validate_and_retry_diagrams.py')

validated_path = REPO_ROOT / 'processed' / 'silver' / 'qa' / 'validated_diagrams.ndjson'
validated_rows = load_ndjson(validated_path)
validated_rows[:2]

## Step 7: Select Final By Angle

In [ ]:
run_stage('select_final_by_angle.py')

selected_path = REPO_ROOT / 'processed' / 'silver' / 'qa' / 'selected_diagrams.ndjson'
selected_rows = load_ndjson(selected_path)
selected_rows[:2]

In [ ]:
run_stage('extract_trajectories.py')

gold_path = REPO_ROOT / 'processed' / 'gold' / 'ndjson' / 'gold_samples.ndjson'
run_stage('export_gold_ndjson.py')
gold_rows = load_ndjson(gold_path)
if gold_rows:
    first = gold_rows[0]
    display(SVG(filename=str(REPO_ROOT / first['gold_refs']['svg_path'])))
    first['trajectory']

## Step 8: Generate Dataset Report

In [ ]:
run_stage('generate_dataset_report.py')

report_json = REPO_ROOT / 'artifacts' / 'reports' / 'dataset_report.json'
report_md = REPO_ROOT / 'artifacts' / 'reports' / 'dataset_report.md'
print(report_json.read_text(encoding='utf-8'))
print('\n---\n')
print(report_md.read_text(encoding='utf-8'))